# Aula 06 — Processamento de Linguagem Natural

Células na ordem dos slides; algumas dependem de variáveis de células anteriores.

## Dados de exemplo (adicionado no notebook)

Os slides usam `reviews` e `tokens_limpos`, que não aparecem nos slides. São reviews sintéticas.

In [ ]:
import re
# [notebook] Dados sintÃ©ticos de exemplo â€” os slides assumem um corpus "jÃ¡ preparado" (reviews + labels)
import random
random.seed(42)
_pos = ["Adorei o produto", "Excelente qualidade", "Entrega rÃ¡pida e bem embalado", "Recomendo muito",
        "Funciona perfeitamente", "Ã“timo atendimento", "Superou minhas expectativas", "Produto maravilhoso"]
_neg = ["PÃ©ssimo produto", "Chegou quebrado", "Entrega atrasada", "NÃ£o recomendo",
        "Veio com defeito", "Atendimento horrÃ­vel", "Qualidade ruim", "Muito decepcionado"]
_neu = ["comprei na semana passada", "o preÃ§o estava bom", "a embalagem era simples", "usei por alguns dias",
        "chegou em casa ontem", "a cor Ã© como na foto"]
reviews_corpus, labels = [], []
for _ in range(400):
    y_ = random.randint(0, 1)
    frases = random.sample(_pos if y_ else _neg, 2) + random.sample(_neu, 1)
    random.shuffle(frases)
    reviews_corpus.append(". ".join(frases) + ".")
    labels.append(y_)
reviews = reviews_corpus  # nome usado na aula 05
print(len(reviews_corpus), "reviews | positivas:", sum(labels))
# tokens já limpos (resultado esperado da Aula 2): minúsculas, sem pontuação nem stop words curtas
_stop = {"o", "a", "e", "de", "na", "no", "um", "uma", "do", "da", "em", "como", "meu", "minha"}
tokens_limpos = [[t for t in re.findall(r"\w+", r.lower()) if t not in _stop] for r in reviews]

## Slide 19 — LDA EM GENSIM — CÓDIGO COMPLETO REVIEWS EM PT-BR, DO BOW AO COHERENCE

*Bloco 01*

In [ ]:
from gensim import corpora
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel
# 'tokens_limpos' = lista de listas de tokens (output Aula 2)
# Ex.: [["produto", "bom"], ["atendimento", "ruim"], ...]
# 1) Criar dicionário e corpus em BoW
dicionario = corpora.Dictionary(tokens_limpos)
dicionario.filter_extremes(no_below=10, no_above=0.5)
corpus_bow = [dicionario.doc2bow(d) for d in tokens_limpos]
# 2) Treinar LDA com 10 tópicos
lda = LdaModel(corpus=corpus_bow, id2word=dicionario, num_topics=10,
               passes=20, alpha="auto", eta="auto", random_state=42)
# 3) Inspecionar tópicos descobertos
for idx, topic in lda.print_topics(num_words=8):
    print(f"Tópico {idx}: {topic}")
# 4) Coerência (qualidade dos tópicos)
coh = CoherenceModel(model=lda, texts=tokens_limpos,
                     dictionary=dicionario, coherence="c_v", processes=1)
print(f"Coherence (c_v): {coh.get_coherence():.3f}")

## Slide 38 — PYLDAVIS — O PADRÃO DE VISUALIZAÇÃO A FERRAMENTA QUE VENDEU LDA A CIENTISTAS DE DADOS

*Bloco 02*

In [ ]:
# pip install pyLDAvis
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
# Preparar visualização (lda, corpus_bow, dicionario do slide anterior)
vis = gensimvis.prepare(lda, corpus_bow, dicionario, mds="mmds")  # [notebook] evita erro de número complexo no PCoA padrão
pyLDAvis.enable_notebook()
pyLDAvis.display(vis)
# Visualização interativa exibida direto no notebook.

> ℹ️ **Adicionado no notebook:** as células de BERTopic baixam o modelo `paraphrase-multilingual-MiniLM-L12-v2` (~470 MB) na primeira execução e levam alguns minutos. Com as reviews sintéticas os tópicos serão pouco interessantes; troque `reviews` pelo seu corpus real.

## Slide 53 — BERTOPIC EM PT-BR — CÓDIGO COMPLETO SETUP COM SENTENCE TRANSFORMER MULTILINGUAL

*Bloco 03*

In [ ]:
# pip install bertopic sentence-transformers
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
# Modelo de embeddings em português (multilingual)
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
# BERTopic com configuração padrão
topic_model = BERTopic(
    embedding_model=embedder,
    language="multilingual",
    min_topic_size=20,        # ignora tópicos com menos de 20 docs
    nr_topics="auto",
)
# 'reviews' é uma lista de strings em PT
topics, probs = topic_model.fit_transform(reviews)
# Inspecionar tópicos
print(topic_model.get_topic_info().head(10))
# Topic  Count                          Name
#    -1   1245   -1_geral_outros_diversos
#     0    832     0_entrega_prazo_atraso
#     1    721     1_qualidade_produto_bom
#     2    615     2_atendimento_suporte_problema

## Slide 54 — BERTOPIC — VISUALIZAÇÕES NATIVAS INSPIRADAS NO PYLDAVIS E MAIS RICAS AINDA

*Bloco 04*

In [ ]:
# Mapa de tópicos em 2D (similar ao pyLDAvis)
fig = topic_model.visualize_topics()
fig.show()
# Barras horizontais com palavras-chave por tópico
fig = topic_model.visualize_barchart(top_n_topics=10)
fig.show()
# Hierarquia de tópicos (dendrograma)
fig = topic_model.visualize_hierarchy(top_n_topics=20)
fig.show()
# Heatmap de similaridade entre tópicos
fig = topic_model.visualize_heatmap(top_n_topics=15)
fig.show()
# Essas visualizações são especialmente úteis em apresentações
# para times de produto, marketing ou C-level que precisam
# entender o que está sendo discutido nos dados.

## Slide 58 — BERTOPIC VS LDA — COMPARAÇÃO DIRETA CÓDIGO DE BENCHMARK EM CORPUS DE REVIEWS PT-BR

*Bloco 05*

In [ ]:
import time
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel
# 1) LDA via Gensim
t0 = time.time()
lda = LdaModel(corpus=corpus_bow, id2word=dicionario,
               num_topics=10, passes=20, random_state=42)
tempo_lda = time.time() - t0
coh_lda = CoherenceModel(model=lda, texts=tokens_limpos,
                         dictionary=dicionario, coherence="c_v", processes=1).get_coherence()
# 2) BERTopic
t0 = time.time()
topic_model = BERTopic(embedding_model=embedder,
                       language="multilingual", min_topic_size=20)
topics, _ = topic_model.fit_transform(reviews)
tempo_bertopic = time.time() - t0
# Coerência manual: top words de cada tópico
top_words_bert = [[w for w, _ in topic_model.get_topic(t)[:10]]
                  for t in topic_model.get_topics() if t != -1]
coh_bert = CoherenceModel(topics=top_words_bert, texts=tokens_limpos,
                          dictionary=dicionario, coherence="c_v", processes=1).get_coherence()
print(f"{'Modelo':<12} {'Tempo (s)':<12} {'Coherence':<12}")
print(f"{'LDA':<12} {tempo_lda:<12.1f} {coh_lda:<12.3f}")
print(f"{'BERTopic':<12} {tempo_bertopic:<12.1f} {coh_bert:<12.3f}")
# Saída em corpus de 10k reviews:
# LDA          45.2          0.428
# BERTopic     312.8         0.567   ← mais coerente, mais lento